In [23]:
import pandas as pd
import numpy as np

In [189]:
gtexEP = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/gtexExpressionProfile.parquet")
emtabEP = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/emtabExpressionProfile.parquet")
orthologTable = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/orthologTable.txt", sep="\t")
orthologTable = orthologTable[~orthologTable["Mouse gene stable ID"].isna()].loc[:, ["Gene stable ID", "Mouse gene stable ID"]]

In [223]:
gtexEP

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type
Gene stable ID,,,,,,,,,
ENSG00000000003,5.799691,22.906008,18.107586,3.414238,15.745596,23.313591,7.128223,10.594314,protein_coding
ENSG00000000005,0.169146,0.753748,0.284215,0.258039,0.956031,0.018972,0.050229,0.206583,protein_coding
ENSG00000000419,21.387617,40.514694,43.498722,24.609381,25.236029,22.603436,22.147223,37.476429,protein_coding
ENSG00000000457,2.830432,6.585192,6.015115,1.902403,3.748399,4.093346,3.044870,5.295540,protein_coding
ENSG00000000460,1.347589,2.231597,2.198975,0.689768,0.988887,1.240150,0.699461,1.596039,protein_coding
...,...,...,...,...,...,...,...,...,...
ENSG00000310553,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA
ENSG00000310554,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA
ENSG00000310555,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA


In [222]:
emtabEP

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type
Gene stable ID,,,,,,,,,
ENSMUSG00000000001,3.491125,12.918058,14.327069,7.289805,7.292047,4.574455,0.890161,5.400340,protein_coding
ENSMUSG00000000003,0.000000,0.000000,0.000000,0.005368,0.000000,0.000000,0.000000,0.000000,protein_coding
ENSMUSG00000000028,0.223612,1.132353,0.561331,2.268044,0.228048,0.087172,0.018164,0.349117,protein_coding
ENSMUSG00000000031,0.256743,0.113783,265.025704,1.737279,0.075890,0.130723,0.167222,0.228744,lncRNA
ENSMUSG00000000037,0.122693,0.267582,0.328520,0.077485,0.033373,0.001122,0.000000,0.028755,protein_coding
...,...,...,...,...,...,...,...,...,...
ENSMUSG00000109574,0.033280,0.004708,0.002177,0.001534,0.000838,0.000000,0.000000,0.000000,TEC
ENSMUSG00000109575,0.109874,0.000000,0.000000,0.000907,0.000000,0.000000,0.000000,0.000000,TEC
ENSMUSG00000109576,0.002251,0.000000,0.000000,0.002440,0.001285,0.000000,0.000000,0.000000,TEC


In [ ]:
orthologTest = tuple(orthologTable.iloc[0, :])
humanOrtholog = gtexEP[gtexEP.index.isin(orthologTest)]
mouseOrtholog = emtabEP[emtabEP.index.isin(orthologTest)]

In [184]:
def euclideanDist(humanOrtholog, mouseOrtholog):
    distSum = 0
    for i in range(0, humanOrtholog.shape[1] - 1):
        distSum += np.square(humanOrtholog.iloc[0, i] - mouseOrtholog.iloc[0, i])
    return np.sqrt(distSum)

In [160]:
euclideanDist(humanOrtholog, mouseOrtholog)

np.float64(73567.9215046818)

In [67]:
# Euclidean Distance
np.linalg.norm(np.array(humanOrtholog.iloc[:, :-1]) - np.array(mouseOrtholog.iloc[:, :-1]))

np.float64(73567.92150468178)

In [185]:
# Pearson Distance
def pearsonDist(humanOrtholog, mouseOrtholog):
    ZxT = ((humanOrtholog.iloc[:, :-1] - np.mean(humanOrtholog.iloc[:, :-1])) / np.std(humanOrtholog.iloc[:, :-1])).T
    Zy = (mouseOrtholog.iloc[:, :-1] - np.mean(mouseOrtholog.iloc[:, :-1])) / np.std(mouseOrtholog.iloc[:, :-1])
    return 1 - ((Zy.dot(ZxT) / ZxT.shape[0]).iloc[0, 0])

In [129]:
pearsonDist(humanOrtholog, mouseOrtholog)

np.float64(0.2094305670992722)

In [146]:
corr = humanOrtholog.iloc[:, :-1].reset_index(drop=True).corrwith(mouseOrtholog.iloc[:, :-1].reset_index(drop=True), method="pearson", axis=1)
1 - corr[0]

np.float64(0.20943055191829996)

In [186]:
# TEC
def TEC(humanOrtholog, mouseOrtholog):
    humanOrthoBinary = humanOrtholog.iloc[:, :-1] > 0
    mouseOrthoBinary = mouseOrtholog.iloc[:, :-1] > 0
    return ((humanOrthoBinary & ~mouseOrthoBinary).sum(axis=1) / humanOrthoBinary.shape[1]).sum() / 2

In [191]:
orthologTest = tuple(orthologTable.iloc[0, :])
humanOrtholog = gtexEP[gtexEP.index.isin(orthologTest)]
mouseOrtholog = emtabEP[emtabEP.index.isin(orthologTest)]

In [199]:
myEuclideanDistArr = []
myPearsonDistArr = []
myTECArr = []
for i in range(0, orthologTable.shape[0]):
    orthologTest = tuple(orthologTable.iloc[i, :])
    humanOrtholog = gtexEP[gtexEP.index.isin(orthologTest)]
    mouseOrtholog = emtabEP[emtabEP.index.isin(orthologTest)]

    if not humanOrtholog.empty and not mouseOrtholog.empty:
        myEuclideanDistArr.append((orthologTest[0], orthologTest[1], euclideanDist(humanOrtholog, mouseOrtholog)))
        myPearsonDistArr.append((orthologTest[0], orthologTest[1], pearsonDist(humanOrtholog, mouseOrtholog)))
        myTECArr.append((orthologTest[0], orthologTest[1], TEC(humanOrtholog, mouseOrtholog)))

In [216]:
myEuclidDistDF = pd.DataFrame(myEuclideanDistArr, columns=["Human ID", "Mouse ID", "EuclidDist"])
myPearDistDF = pd.DataFrame(myPearsonDistArr, columns=["Human ID", "Mouse ID", "PearDist"])
myTECDF = pd.DataFrame(myTECArr, columns=["Human ID", "Mouse ID", "TEC"])

In [209]:
myEuclidDistDF.loc[:, ["Human ID", "EuclidDist"]].groupby("Human ID").mean().reset_index()

,Human ID,EuclidDist
0,ENSG00000000003,39.108790
1,ENSG00000000005,1.249876
2,ENSG00000000419,68.410490
3,ENSG00000000457,8.682555
4,ENSG00000000460,3.533564
...,...,...
17466,ENSG00000293570,3.972581
17467,ENSG00000293662,0.423517
17468,ENSG00000293663,0.042750
17469,ENSG00000300510,30.267226


In [224]:
gtexEPEuclid = gtexEP.merge(myEuclidDistDF.loc[:, ["Human ID", "EuclidDist"]].groupby("Human ID").mean().reset_index(), left_index=True, right_on="Human ID", how="outer").set_index("Human ID")
gtexEPEuclidPear = gtexEPEuclid.merge(myPearDistDF.loc[:, ["Human ID", "PearDist"]].groupby("Human ID").mean().reset_index(), left_index=True, right_on="Human ID", how="outer").set_index("Human ID")
gtexEPEuclidPearTEC = gtexEPEuclidPear.merge(myTECDF.loc[:, ["Human ID", "TEC"]].groupby("Human ID").mean().reset_index(), left_index=True, right_on="Human ID", how="outer").set_index("Human ID")
gtexEPEuclidPearTEC.to_csv("/Users/andrewhsu/Projects/McNair/data/gtexEPDist.csv", index=True)
gtexEPEuclidPearTEC.to_parquet("/Users/andrewhsu/Projects/McNair/data/gtexEPDist.parquet", index=True)

In [221]:
gtexEPEuclidPearTEC

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type,EuclidDist,PearDist,TEC
Human ID,,,,,,,,,,,,
ENSG00000000003,5.799691,22.906008,18.107586,3.414238,15.745596,23.313591,7.128223,10.594314,protein_coding,39.108790,1.130272,0.0
ENSG00000000005,0.169146,0.753748,0.284215,0.258039,0.956031,0.018972,0.050229,0.206583,protein_coding,1.249876,1.141118,0.0
ENSG00000000419,21.387617,40.514694,43.498722,24.609381,25.236029,22.603436,22.147223,37.476429,protein_coding,68.410490,1.211158,0.0
ENSG00000000457,2.830432,6.585192,6.015115,1.902403,3.748399,4.093346,3.044870,5.295540,protein_coding,8.682555,0.814870,0.0
ENSG00000000460,1.347589,2.231597,2.198975,0.689768,0.988887,1.240150,0.699461,1.596039,protein_coding,3.533564,0.959842,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000310553,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA,NaN,NaN,NaN
ENSG00000310554,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA,NaN,NaN,NaN
ENSG00000310555,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA,NaN,NaN,NaN


In [225]:
emtabEPEuclid = emtabEP.merge(myEuclidDistDF.loc[:, ["Mouse ID", "EuclidDist"]].groupby("Mouse ID").mean().reset_index(), left_index=True, right_on="Mouse ID", how="outer").set_index("Mouse ID")
emtabEPEuclidPear = emtabEPEuclid.merge(myPearDistDF.loc[:, ["Mouse ID", "PearDist"]].groupby("Mouse ID").mean().reset_index(), left_index=True, right_on="Mouse ID", how="outer").set_index("Mouse ID")
emtabEPEuclidPearTEC = emtabEPEuclidPear.merge(myTECDF.loc[:, ["Mouse ID", "TEC"]].groupby("Mouse ID").mean().reset_index(), left_index=True, right_on="Mouse ID", how="outer").set_index("Mouse ID")
emtabEPEuclidPearTEC.to_csv("/Users/andrewhsu/Projects/McNair/data/emtabEPDist.csv", index=True)
emtabEPEuclidPearTEC.to_parquet("/Users/andrewhsu/Projects/McNair/data/emtabEPDist.parquet", index=True)

In [220]:
emtabEPEuclidPearTEC

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type,EuclidDist,PearDist,TEC
Mouse ID,,,,,,,,,,,,
ENSMUSG00000000001,3.491125,12.918058,14.327069,7.289805,7.292047,4.574455,0.890161,5.400340,protein_coding,15.649925,0.151442,0.0
ENSMUSG00000000003,0.000000,0.000000,0.000000,0.005368,0.000000,0.000000,0.000000,0.000000,protein_coding,NaN,NaN,NaN
ENSMUSG00000000028,0.223612,1.132353,0.561331,2.268044,0.228048,0.087172,0.018164,0.349117,protein_coding,4.136989,1.001638,0.0
ENSMUSG00000000031,0.256743,0.113783,265.025704,1.737279,0.075890,0.130723,0.167222,0.228744,lncRNA,NaN,NaN,NaN
ENSMUSG00000000037,0.122693,0.267582,0.328520,0.077485,0.033373,0.001122,0.000000,0.028755,protein_coding,3.303883,0.161786,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
ENSMUSG00000109574,0.033280,0.004708,0.002177,0.001534,0.000838,0.000000,0.000000,0.000000,TEC,NaN,NaN,NaN
ENSMUSG00000109575,0.109874,0.000000,0.000000,0.000907,0.000000,0.000000,0.000000,0.000000,TEC,NaN,NaN,NaN
ENSMUSG00000109576,0.002251,0.000000,0.000000,0.002440,0.001285,0.000000,0.000000,0.000000,TEC,NaN,NaN,NaN
